# Z5008 Big Data Lab — Lab 6: Apache Airflow & Pipeline Orchestration

**Instructor:** Dr. Innocent Nyalala · `innocent@iitmz.ac.in`  
**Institution:** IIT Madras Zanzibar  
**Date:** Monday, 27 April 2026  
**Assignment due:** Sunday, 3 May 2026, 23:59 EAT

---

## What is Apache Airflow?

Airflow is a **workflow orchestration** platform. Instead of running Python scripts manually, you define a **DAG** (Directed Acyclic Graph) that describes:
- **What** tasks to run (Python functions, Bash commands, Spark jobs, …)
- **In what order** they must run (task dependencies)
- **When** to run them (a schedule, or manually)

Airflow then handles scheduling, retrying failed tasks, logging every run, and showing you a visual dashboard.

## Service URLs (open in your browser)

| Service | URL | Login |
|---------|-----|-------|
| JupyterLab | http://localhost:8888 | token: `bigdata` |
| **Airflow UI** | **http://localhost:8090** | **admin / admin** |
| Spark Master UI | http://localhost:8080 | — |
| MinIO Console | http://localhost:9001 | admin / bigdata123 |

## Learning Objectives
1. Verify the full lab environment (Docker, Airflow, Spark, MinIO)
2. Upload data to MinIO from Python
3. Write DAG files programmatically from this notebook
4. Trigger DAG runs using the Airflow REST API
5. Monitor task states and retrieve logs via the REST API
6. Write and submit a PySpark job that reads from and writes to MinIO

---
## Section 0 — Setup: Install Libraries & Global Configuration

**Run this section first, every time you open this notebook.**  
It installs missing libraries and sets all hostnames/credentials in one place.

In [1]:
# ── Install boto3 (the Amazon S3 / MinIO Python SDK) ────────────────────────────
# boto3 is not pre-installed in the Jupyter image, so we install it here.
# subprocess lets us run shell commands from inside Python.
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "boto3", "--quiet"],  # sys.executable = path to this notebook's Python
    check=True,           # check=True = raise an error and stop if the install fails
)
print("boto3 installed successfully — ready to proceed.")

boto3 installed successfully — ready to proceed.


In [2]:
# ── Global Configuration — all hostnames and credentials in ONE place ───────────
#
# IMPORTANT CONCEPT — Docker internal vs host addresses:
#   Inside Docker, every container can reach every other container by its SERVICE NAME.
#   Example: the Jupyter container reaches Airflow at 'airflow:8080' (internal port).
#   From YOUR laptop's browser you use localhost:PORT (the mapped host port).
#
#   Service    | From inside Docker   | From your browser
#   -----------|----------------------|-------------------
#   Airflow    | http://airflow:8080  | http://localhost:8090
#   MinIO API  | http://minio:9000    | http://localhost:9000
#   MinIO UI   | (not needed)         | http://localhost:9001

# ── Airflow ──────────────────────────────────────────────────────────────────────
AIRFLOW_BASE = "http://airflow:8080"   # container name 'airflow' + internal port 8080 (mapped to localhost:8090)
AIRFLOW_AUTH = ("admin", "admin")      # credentials created by airflow-init service in docker-compose.yml

# ── MinIO (S3-compatible object storage) ─────────────────────────────────────────
MINIO_ENDPOINT = "http://minio:9000"   # container name 'minio' + S3 API port 9000
MINIO_ACCESS   = "admin"               # matches MINIO_ROOT_USER in docker-compose.yml
MINIO_SECRET   = "bigdata123"          # matches MINIO_ROOT_PASSWORD in docker-compose.yml
MINIO_BUCKET   = "warehouse"           # the pre-created S3 bucket for all lab data

# ── Shared folder paths (the key to fixing the PermissionError) ──────────────────
# docker-compose mounts the SAME HOST FOLDER into multiple containers under different paths:
#
#   Host folder: ./dags/
#   In Jupyter:           /home/jovyan/work/dags/    ← we write DAG files here
#   In Airflow webserver: /opt/airflow/dags/          ← Airflow reads DAG files here
#   In Airflow scheduler: /opt/airflow/dags/          ← scheduler watches this folder
#
# Because it is the SAME physical folder, any file we write from Jupyter
# immediately appears in Airflow's DAG folder — no manual copying needed!

DAGS_DIR       = "/home/jovyan/work/dags"        # Jupyter's path to the shared DAGs folder
SPARK_JOBS_DIR = "/home/jovyan/work/spark_jobs"  # Jupyter's path to the shared Spark jobs folder

DATE = "2026-04-27"   # processing date used in file paths — update this each lab session

import os
os.makedirs(DAGS_DIR, exist_ok=True)        # create the folder if it does not exist yet
os.makedirs(SPARK_JOBS_DIR, exist_ok=True)  # same for spark_jobs

print(f"Airflow URL    : {AIRFLOW_BASE}  (browser: http://localhost:8090)")
print(f"MinIO endpoint : {MINIO_ENDPOINT}")
print(f"DAGs folder    : {DAGS_DIR}")
print(f"Spark jobs     : {SPARK_JOBS_DIR}")
print(f"Processing date: {DATE}")

Airflow URL    : http://airflow:8080  (browser: http://localhost:8090)
MinIO endpoint : http://minio:9000
DAGs folder    : /home/jovyan/work/dags
Spark jobs     : /home/jovyan/work/spark_jobs
Processing date: 2026-04-27


In [3]:
# ── Spark Session Setup ──────────────────────────────────────────────────────────
# We use local[*] mode (all CPU cores on this machine) so no external Spark cluster
# is needed for the in-notebook tests. The Airflow DAG (Section 5) will use the
# real Spark cluster via SparkSubmitOperator.

import urllib.request   # urllib.request = download files from the internet

cwd       = os.getcwd()  # get the current working directory where we will save JARs
jar1_path = os.path.join(cwd, "hadoop-aws-3.3.4.jar")             # path for JAR 1
jar2_path = os.path.join(cwd, "aws-java-sdk-bundle-1.12.262.jar") # path for JAR 2

# These two JARs teach Spark how to read/write files at s3a://... (MinIO / Amazon S3)
print("Checking JAR files (download once, reuse after)...")
if not os.path.exists(jar1_path):   # only download if the file is not already on disk
    print("  Downloading hadoop-aws-3.3.4.jar ...")
    urllib.request.urlretrieve(
        "https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar",
        jar1_path
    )
if not os.path.exists(jar2_path):   # same check for the second JAR
    print("  Downloading aws-java-sdk-bundle-1.12.262.jar ...")
    urllib.request.urlretrieve(
        "https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar",
        jar2_path
    )
print("  JARs ready.")

# Inject the JARs into Spark BEFORE it starts (this environment variable is read at startup)
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    f"--jars {jar1_path},{jar2_path} pyspark-shell"  # comma-separated JAR paths
)

from pyspark.sql import SparkSession   # SparkSession = main entry point to the Spark engine

# Kill any leftover SparkContext from Lab 05 (causes 'stopped SparkContext' error if not cleared)
try:
    SparkSession.builder.getOrCreate().stop()  # .stop() gracefully shuts down the existing session
    print("Previous Spark session stopped.")
except Exception:
    pass   # no previous session — nothing to stop, continue normally

spark = (
    SparkSession.builder
    .appName("Lab06-Airflow")
    .master("local[*]")
    # --- ADD THIS LINE TO DISABLE THE LOGS ---
    .config("spark.eventLog.enabled", "false") 
    # ------------------------------------------
    .config("spark.hadoop.fs.s3a.endpoint",           MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",         MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",         MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access",  "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")   # hide noisy INFO/WARN messages; only show real errors
print(f"Spark version : {spark.version}")
print(f"Master        : {spark.sparkContext.master}")

Checking JAR files (download once, reuse after)...
  JARs ready.
Spark version : 3.5.0
Master        : local[*]


---
## Section 1 — Environment Verification

Confirm all Docker services are running before proceeding.

In [4]:
# ── Verify all services are healthy via their HTTP endpoints ────────────────────
# NOTE: The Jupyter container runs INSIDE Docker and does NOT have the Docker CLI.
# We cannot run 'docker compose ps' from here.
# Instead, we call each service's own built-in health endpoint over the internal network.
# This is more useful anyway — it tells us if the service is ACTUALLY ready, not just 'started'.

import requests   # requests = standard Python library for making HTTP calls

services = [
    # (display label,            URL to check,                              auth tuple or None)
    ('Airflow webserver',  f'{AIRFLOW_BASE}/api/v1/health',              AIRFLOW_AUTH),
    ('MinIO S3 API    ',   'http://minio:9000/minio/health/ready',        None),
    ('Spark Master UI ',   'http://spark-master:8080',                    None),
    ('PostgreSQL proxy',   f'{AIRFLOW_BASE}/api/v1/health',              AIRFLOW_AUTH),  # Airflow health includes DB
]

print('Service health check:')
all_ok = True
for label, url, auth in services:
    try:
        kw = {'timeout': 5}                    # give up after 5 seconds if no response
        if auth:
            kw['auth'] = auth                  # attach Basic Auth credentials where required
        resp = requests.get(url, **kw)
        ok   = resp.status_code < 400          # any 2xx or 3xx = service is up
        icon = 'OK ' if ok else 'WARN'
        print(f'  [{icon}]  {label}  HTTP {resp.status_code}')
        if not ok:
            all_ok = False
    except Exception as e:
        print(f'  [ERR]  {label}  {type(e).__name__}: {e}')
        all_ok = False

print()
if all_ok:
    print('All services reachable — ready to proceed.')
else:
    print('One or more services not reachable.')
    print('Make sure Docker Desktop is running and the stack is up:')
    print('  cd D:\Big_Data_Lab\infrastructure  &&  docker-compose up -d')


Service health check:
  [OK ]  Airflow webserver  HTTP 200
  [OK ]  MinIO S3 API      HTTP 200
  [OK ]  Spark Master UI   HTTP 200
  [OK ]  PostgreSQL proxy  HTTP 200

All services reachable — ready to proceed.


In [5]:
# ── Verify Airflow is healthy via its REST API ──────────────────────────────────
# Airflow exposes a REST API at /api/v1/ that lets us:
#   - list DAGs, trigger runs, check task states, fetch logs
# All without touching the Airflow UI at all.

import requests   # requests = the standard Python library for making HTTP calls
import json       # json = convert between Python dicts and JSON text

resp = requests.get(
    f"{AIRFLOW_BASE}/api/v1/health",   # /api/v1/health = Airflow's built-in health-check endpoint
    auth=AIRFLOW_AUTH,                  # auth=(username, password) → HTTP Basic Authentication
    timeout=10,                         # give up after 10 seconds if no response
)

print(f"HTTP Status : {resp.status_code}")   # 200 = OK; 401 = wrong password; 503 = Airflow starting up
print(json.dumps(resp.json(), indent=2))      # pretty-print the response — look for 'status: healthy'

HTTP Status : 200
{
  "dag_processor": {
    "latest_dag_processor_heartbeat": null,
    "status": null
  },
  "metadatabase": {
    "status": "healthy"
  },
  "scheduler": {
    "latest_scheduler_heartbeat": "2026-04-27T13:17:31.024415+00:00",
    "status": "healthy"
  },
  "triggerer": {
    "latest_triggerer_heartbeat": null,
    "status": null
  }
}


---
## Section 2 — Upload Sample Data to MinIO

We generate 1,000 synthetic mobile-money transactions and upload them to MinIO at:  
`s3a://warehouse/raw/transactions/DATE/data.csv`

This simulates the raw data that our Airflow ETL pipeline will process.

In [6]:
# ── Connect to MinIO and create the storage bucket ──────────────────────────────
# boto3 is Amazon's official Python SDK. Because MinIO is S3-compatible,
# boto3 works with MinIO out of the box — we just point it at our server instead of AWS.

import boto3                              # boto3 = Python SDK for Amazon S3 (and S3-compatible storage)
from botocore.exceptions import ClientError   # ClientError = how boto3 reports S3 errors

s3 = boto3.client(
    "s3",                                 # 's3' = use the Simple Storage Service interface
    endpoint_url=MINIO_ENDPOINT,          # override the default AWS endpoint with our MinIO address
    aws_access_key_id=MINIO_ACCESS,       # MinIO username (from our global config)
    aws_secret_access_key=MINIO_SECRET,   # MinIO password (from our global config)
    region_name="us-east-1",              # boto3 requires a region; any value works for MinIO
)

try:
    s3.create_bucket(Bucket=MINIO_BUCKET)   # create_bucket() = make a new top-level storage container
    print(f"Bucket '{MINIO_BUCKET}' created.")
except ClientError as e:
    code = e.response["Error"]["Code"]       # extract the error code string from the exception
    if code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):  # bucket exists from a previous run
        print(f"Bucket '{MINIO_BUCKET}' already exists — OK, continuing.")
    else:
        raise   # unexpected error — re-raise so we see the full traceback

Bucket 'warehouse' already exists — OK, continuing.


In [7]:
# ── Generate synthetic transaction data and upload to MinIO ─────────────────────
import random   # random = generate random numbers (used to create fake transaction data)
import io       # io = work with in-memory streams (file-like objects that live in RAM)
import csv      # csv = read and write Comma-Separated Values files

random.seed(42)   # seed = makes 'random' numbers reproducible; same seed = same data every run

COUNTRIES = ["KE", "TZ", "UG", "RW", "ET"]          # ISO country codes for 5 African markets
STATUSES  = ["SUCCESS", "FAILED", "PENDING"]         # possible transaction outcomes
N_ROWS    = 1_000                                     # how many fake transactions to generate

fieldnames = ["txn_id", "amount", "sender", "receiver", "country", "status", "txn_date"]

rows = []
for i in range(N_ROWS):
    rows.append({
        "txn_id"   : f"TXN{i:07d}",                                     # zero-padded 7-digit ID
        "amount"   : round(random.uniform(50, 100_000), 2),             # random amount, 2 decimal places
        "sender"   : f"254{random.randint(700_000_000, 799_999_999)}",  # fake Kenyan (+254) phone number
        "receiver" : f"255{random.randint(700_000_000, 799_999_999)}",  # fake Tanzanian (+255) phone number
        "country"  : random.choice(COUNTRIES),                          # random country for each row
        "status"   : random.choices(STATUSES, weights=[0.75, 0.15, 0.10])[0],  # 75% success, 15% failed, 10% pending
        "txn_date" : DATE,                                              # use today's processing date
    })

# Build the CSV content in memory (avoids creating a temp file on disk)
buf = io.StringIO()                                      # StringIO = an in-memory text buffer
writer = csv.DictWriter(buf, fieldnames=fieldnames)      # DictWriter = writes Python dicts as CSV rows
writer.writeheader()                                     # write the column name row first
writer.writerows(rows)                                   # write all 1,000 data rows

# Upload to MinIO — the path becomes: s3a://warehouse/raw/transactions/DATE/data.csv
KEY = f"raw/transactions/{DATE}/data.csv"                # S3 'key' = full path inside the bucket
s3.put_object(
    Bucket=MINIO_BUCKET,              # which bucket to write to
    Key=KEY,                          # the path within the bucket
    Body=buf.getvalue().encode(),     # .getvalue() = get the string; .encode() = convert to bytes
)
print(f"Uploaded {N_ROWS:,} rows  ->  s3a://{MINIO_BUCKET}/{KEY}")

Uploaded 1,000 rows  ->  s3a://warehouse/raw/transactions/2026-04-27/data.csv


In [8]:
# ── Verify the upload by reading the file back through Spark ────────────────────
df = spark.read.csv(
    f"s3a://{MINIO_BUCKET}/raw/transactions/{DATE}/data.csv",
    header=True,       # first row of the CSV contains column names
    inferSchema=True,  # automatically detect column data types (e.g. amount as double)
)

print(f"Row count : {df.count()}")    # should be exactly 1,000
print(f"Columns   : {df.columns}")   # should list our 7 field names

# Cross-tabulation: count rows grouped by (country, status) to see the data distribution
df.groupBy("country", "status") \
  .count() \
  .orderBy("country", "status") \
  .show()

Row count : 1000
Columns   : ['txn_id', 'amount', 'sender', 'receiver', 'country', 'status', 'txn_date']
+-------+-------+-----+
|country| status|count|
+-------+-------+-----+
|     ET| FAILED|   30|
|     ET|PENDING|   15|
|     ET|SUCCESS|  170|
|     KE| FAILED|   29|
|     KE|PENDING|   16|
|     KE|SUCCESS|  150|
|     RW| FAILED|   30|
|     RW|PENDING|   32|
|     RW|SUCCESS|  143|
|     TZ| FAILED|   29|
|     TZ|PENDING|   13|
|     TZ|SUCCESS|  142|
|     UG| FAILED|   35|
|     UG|PENDING|   23|
|     UG|SUCCESS|  143|
+-------+-------+-----+



---
## Section 3 — Write DAG Files from the Notebook

**How DAG discovery works:**  
The Airflow Scheduler continuously watches the `/opt/airflow/dags/` folder.  
Any `.py` file placed there is automatically parsed and registered as a DAG within ~30 seconds.

Our `docker-compose.yml` mounts the **same host folder** into both Jupyter (`/home/jovyan/work/dags/`)  
and Airflow (`/opt/airflow/dags/`). So writing a file from this notebook is enough — Airflow picks it up automatically.

### The Three DAG Patterns We Will Build

| DAG | Concept taught |
|-----|----------------|
| `lab06_hello_dag` | Basic operators: PythonOperator + BashOperator, sequential dependencies |
| `lab06_taskflow_demo` | TaskFlow API: Python decorators, XCom via return values |
| `lab06_branch_xcom` | BranchPythonOperator, XCom push/pull, conditional paths |

In [9]:
# ── DAG 1: Hello DAG — the simplest possible pipeline ───────────────────────────
# Two tasks, run in sequence: a Python function, then a Bash command.
# Concept: operators, task dependencies, execution context.

hello_dag_code = """
# lab06_hello_dag.py  --  AUTO-GENERATED by lab06_airflow.ipynb
#
# DAG = Directed Acyclic Graph: a set of tasks connected by dependency arrows.
# 'Directed'  = arrows go one way (no loops allowed)
# 'Acyclic'   = no task depends on itself (no cycles)
# 'Graph'     = a network of nodes (tasks) and edges (dependencies)

from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator   # PythonOperator: run any Python function as a task
from airflow.operators.bash import BashOperator       # BashOperator:   run a shell command as a task

# 'with DAG(...) as dag:' is a context manager.
# Every operator created INSIDE this block is automatically registered to this DAG.
with DAG(
    dag_id='lab06_hello_dag',          # unique name shown in the Airflow UI DAG list
    schedule_interval=None,            # None = manual trigger only; no automatic schedule
    start_date=datetime(2026, 4, 1),   # Airflow requires a start_date to know when runs are valid
    catchup=False,                     # False = don't create runs for all past dates; start fresh
    tags=['lab06', 'intro'],           # tags appear as filter chips in the Airflow UI
) as dag:

    def greet(**context):              # **context = dict of Airflow metadata Airflow passes in at runtime
        ds = context['ds']             # 'ds' = execution date string in YYYY-MM-DD format
        print(f'Hello from Airflow! Execution date: {ds}')   # this text appears in the task log
        return {'greeted': True, 'date': ds}                 # return value is auto-stored as XCom

    t1 = PythonOperator(
        task_id='python_greet',        # task_id must be unique within the DAG
        python_callable=greet,         # pass the function object (no parentheses = don't call it now)
    )

    t2 = BashOperator(
        task_id='bash_echo',
        bash_command='echo "Bash task ran on {{ ds }}"',   # {{ ds }} is a Jinja2 template filled in at runtime
    )

    t1 >> t2   # >> = task dependency: t1 must SUCCEED before t2 is allowed to start
"""

dag_path = os.path.join(DAGS_DIR, "lab06_hello_dag.py")   # build the full file path
with open(dag_path, "w") as f:   # "w" mode = create the file (or overwrite if it exists)
    f.write(hello_dag_code)

print(f"Written : {dag_path}")
print("Airflow Scheduler will detect this file within ~30 seconds.")
print("Then open http://localhost:8090 to see it in the DAG list.")

Written : /home/jovyan/work/dags/lab06_hello_dag.py
Airflow Scheduler will detect this file within ~30 seconds.
Then open http://localhost:8090 to see it in the DAG list.


In [10]:
# ── DAG 2: TaskFlow API — modern Airflow with decorators ────────────────────────
# The TaskFlow API (introduced in Airflow 2.0) lets you write DAGs as plain Python
# functions decorated with @task. Dependencies are inferred from function call order.
# XCom (cross-communication) between tasks is handled automatically via return values.

taskflow_dag_code = """
# lab06_taskflow_demo.py  --  AUTO-GENERATED by lab06_airflow.ipynb
#
# TaskFlow API: use @dag and @task decorators instead of manually creating operators.
# XCom: when a @task function returns a value, Airflow automatically stores it.
# The next @task receives it as a function argument — no manual xcom_push/pull needed.

from airflow.decorators import dag, task   # @dag = mark a function as a DAG; @task = mark a function as a task
from datetime import datetime

@dag(
    dag_id='lab06_taskflow_demo',
    schedule_interval=None,
    start_date=datetime(2026, 4, 1),
    catchup=False,
    tags=['lab06', 'taskflow'],
)
def taskflow_demo():
    'Demonstrate the TaskFlow API: extract -> transform -> load with automatic XCom.'

    @task                            # @task = this function becomes an Airflow task automatically
    def extract() -> dict:           # return type hint (dict) tells Airflow how to serialize the XCom value
        'Simulate reading file metadata from an upstream system.'
        return {'rows': 1000, 'source': 'minio', 'date': '2026-04-27'}  # this dict is auto-stored as XCom

    @task
    def transform(raw: dict) -> dict:   # raw = the dict returned by extract() — Airflow pulls XCom automatically
        'Filter and enrich: compute success/failure row counts.'
        raw['successful_rows'] = int(raw['rows'] * 0.75)           # 75% of transactions succeed
        raw['failed_rows']     = raw['rows'] - raw['successful_rows']  # the rest failed
        return raw                                                   # pass enriched dict to next task via XCom

    @task
    def load(data: dict):    # data = the dict returned by transform() — again pulled from XCom automatically
        'Log final stats. In production this would write to a database or Delta table.'
        print(f'[LOAD] Date             : {data["date"]}')
        print(f'[LOAD] Successful rows  : {data["successful_rows"]}/{data["rows"]}')
        print(f'[LOAD] Failed rows      : {data["failed_rows"]}/{data["rows"]}')

    # Call the task functions as if they are normal Python functions.
    # Airflow infers the dependencies from the call order: extract -> transform -> load.
    raw  = extract()
    proc = transform(raw)   # 'raw' is an XCom reference, not the actual dict yet
    load(proc)

dag_instance = taskflow_demo()   # calling the @dag-decorated function registers it with Airflow
"""

dag_path = os.path.join(DAGS_DIR, "lab06_taskflow_demo.py")
with open(dag_path, "w") as f:
    f.write(taskflow_dag_code)
print(f"Written : {dag_path}")

Written : /home/jovyan/work/dags/lab06_taskflow_demo.py


In [11]:
# ── DAG 3: Branching + XCom — conditional pipeline paths ───────────────────────
# Real pipelines often need to take different paths based on data values.
# BranchPythonOperator picks which downstream task to run based on a Python function's return value.
# XCom (cross-communication) is the mechanism tasks use to share small data values.

branch_dag_code = """
# lab06_branch_xcom.py  --  AUTO-GENERATED by lab06_airflow.ipynb

from datetime import datetime, timedelta
from airflow import DAG
from airflow.operators.python import PythonOperator, BranchPythonOperator   # BranchPythonOperator = conditional fork
from airflow.operators.dummy import DummyOperator   # DummyOperator = placeholder task that does nothing (used for join points)

with DAG(
    dag_id='lab06_branch_xcom',
    schedule_interval=None,
    start_date=datetime(2026, 4, 1),
    catchup=False,
    default_args={                     # default_args = settings applied to ALL tasks in this DAG
        'retries': 2,                  # retry a failed task up to 2 times before marking it as failed
        'retry_delay': timedelta(minutes=2),   # wait 2 minutes between retry attempts
        'email_on_failure': False,     # don't try to send email (we haven't configured a mail server)
    },
    tags=['lab06', 'xcom', 'branch'],
) as dag:

    def count_rows(**context):
        'Count ingested rows and push the result to XCom so other tasks can read it.'
        ti        = context['task_instance']   # task_instance = the running task object
        row_count = 1500                       # in production: query MinIO or a database

        # xcom_push stores a key-value pair that any downstream task in the same DAG run can read
        ti.xcom_push(key='row_count',  value=row_count)
        ti.xcom_push(key='quality_ok', value=row_count > 500)   # True if more than 500 rows
        print(f'Row count: {row_count}')

    def choose_branch(**context):
        'Read the row count from XCom and decide which downstream path to take.'
        ti    = context['ti']   # ti = shorthand alias for task_instance
        count = ti.xcom_pull(task_ids='count_rows', key='row_count')  # pull the value pushed by count_rows
        print(f'Branching on count={count}')
        # The return value is the task_id (or list of task_ids) that should run next.
        # All other downstream tasks are automatically SKIPPED.
        return 'high_volume' if count > 1000 else 'low_volume'

    def log_xcom(**context):
        'Demonstrate xcom_pull: read and print both values pushed by count_rows.'
        ti   = context['ti']
        rows = ti.xcom_pull(task_ids='count_rows', key='row_count')
        ok   = ti.xcom_pull(task_ids='count_rows', key='quality_ok')
        print(f'[LOG] row_count={rows}, quality_ok={ok}')

    # Create task objects from the functions above
    count  = PythonOperator(task_id='count_rows',   python_callable=count_rows)
    log    = PythonOperator(task_id='log_xcom',     python_callable=log_xcom)

    # BranchPythonOperator: runs choose_branch() and skips all branches NOT returned
    branch = BranchPythonOperator(task_id='branch', python_callable=choose_branch)

    high   = DummyOperator(task_id='high_volume')   # placeholder for 'send alert: high volume'
    low    = DummyOperator(task_id='low_volume')    # placeholder for 'send alert: low volume'

    # 'none_failed_min_one_success' = this task runs as long as at least one upstream task succeeded
    # (needed because the branch operator skips one path, which would otherwise block the join)
    join   = DummyOperator(task_id='join', trigger_rule='none_failed_min_one_success')

    # Define the dependency graph:
    count >> log >> branch >> [high, low]   # branch fans out to both; only one will actually run
    high  >> join                           # both paths converge at join
    low   >> join
"""

dag_path = os.path.join(DAGS_DIR, "lab06_branch_xcom.py")
with open(dag_path, "w") as f:
    f.write(branch_dag_code)
print(f"Written : {dag_path}")
print("All 3 DAG files written. Wait ~30 seconds, then run Section 4.")

import time, os

# ── Diagnostic: verify the shared volume is working ──────────────────────────────
# classroom_smoke_test.py should already exist in Airflow's DAGs folder.
# If we can see it from here, our DAGS_DIR path is correct and files will reach Airflow.
probe = os.path.join(DAGS_DIR, 'classroom_smoke_test.py')
if os.path.exists(probe):
    print(f'Volume mount OK — classroom_smoke_test.py is visible at {DAGS_DIR}')
else:
    print(f'Volume mount NOT active — classroom_smoke_test.py NOT found at {DAGS_DIR}')
    print('Files are going to the wrong location.')
    print('Fix: run  docker-compose up -d --force-recreate jupyter  from the infrastructure folder,')
    print('     then restart this notebook and re-run from Section 0.')

print(f'\nFiles now in DAGS_DIR:')
for f in sorted(os.listdir(DAGS_DIR)):
    print(f'  {f}')

# ── Wait for the Airflow Scheduler to detect the new files ───────────────────────
# The scheduler polls the DAGs folder every ~30 seconds.
# We sleep 40 seconds here so the scheduler has time to parse and register the DAGs.
print('\nWaiting 40 seconds for the Airflow Scheduler to detect the new DAG files...')
for i in range(4):
    time.sleep(10)                    # sleep 10 seconds at a time so we can see progress
    print(f'  {(i+1)*10}s elapsed...')
print('Ready — now run Section 4.')


Written : /home/jovyan/work/dags/lab06_branch_xcom.py
All 3 DAG files written. Wait ~30 seconds, then run Section 4.
Volume mount NOT active — classroom_smoke_test.py NOT found at /home/jovyan/work/dags
Files are going to the wrong location.
Fix: run  docker-compose up -d --force-recreate jupyter  from the infrastructure folder,
     then restart this notebook and re-run from Section 0.

Files now in DAGS_DIR:
  lab06_branch_xcom.py
  lab06_hello_dag.py
  lab06_taskflow_demo.py

Waiting 40 seconds for the Airflow Scheduler to detect the new DAG files...
  10s elapsed...
  20s elapsed...
  30s elapsed...
  40s elapsed...
Ready — now run Section 4.


---
## Section 4 — Trigger DAGs via the Airflow REST API

**Wait ~30 seconds** after writing the DAG files, then run these cells.  
The Scheduler needs time to parse and register the new DAGs.

The REST API lets us automate everything the Airflow UI can do:  
list DAGs, trigger runs, check status, fetch logs — all from Python.

In [12]:
# ── List all DAGs that Airflow has registered ────────────────────────────────────
# After writing DAG files, the Scheduler parses them and registers them in the database.
# This endpoint shows us all registered DAGs and whether each one is paused or active.

resp = requests.get(
    f'{AIRFLOW_BASE}/api/v1/dags',   # /api/v1/dags = list all DAGs
    auth=AIRFLOW_AUTH,
    timeout=10,
)

# Always check the HTTP status before parsing JSON — a 403 gives JSON with no 'total_entries'
if resp.status_code == 403:
    print('ERROR 403 Forbidden — Basic Auth is not enabled in Airflow.')
    print('Fix: restart the airflow container after adding AIRFLOW__API__AUTH_BACKENDS to docker-compose.yml')
    print('     docker-compose restart airflow      (run this from the infrastructure folder on your laptop)')
elif resp.status_code != 200:
    print(f'ERROR {resp.status_code}: {resp.text}')
else:
    import json as _json
    data = resp.json()   # parse the JSON response body into a Python dict

    print(f'Total DAGs registered: {data.get("total_entries", "?")}')
    for dag in data.get('dags', []):   # 'dags' is a list of DAG info dicts
        status = 'PAUSED' if dag['is_paused'] else 'ACTIVE'   # new DAGs start paused for safety
        print(f'  [{status:6s}]  {dag["dag_id"]}')

    print('\nNote: New DAGs appear as PAUSED. Run the next cell to activate them.')


Total DAGs registered: 0

Note: New DAGs appear as PAUSED. Run the next cell to activate them.


In [13]:
# ── DAG Discovery Diagnostic ─────────────────────────────────────────────────────
import os, requests

# 1. What files exist in DAGS_DIR?
print(f'DAGS_DIR = {DAGS_DIR}')
print('Files found:')
for fname in sorted(os.listdir(DAGS_DIR)):
    size = os.path.getsize(os.path.join(DAGS_DIR, fname))
    print(f'  {fname}  ({size} bytes)')

# 2. Volume mount check — classroom_smoke_test.py must be visible if the bind-mount is active
probe = os.path.join(DAGS_DIR, 'classroom_smoke_test.py')
print()
if os.path.exists(probe):
    print('VOLUME MOUNT: OK  classroom_smoke_test.py is visible here')
    print('  Files ARE reaching Airflow — problem is a parse/import error (see item 3)')
else:
    print('VOLUME MOUNT: BROKEN  classroom_smoke_test.py NOT found in DAGS_DIR')
    print('  Files written here go to ./notebooks/dags/ on the host,')
    print('  but Airflow watches ./dags/ — a completely different folder.')
    print('  FIX: run from D:\Big_Data_Lab\infrastructure on your laptop:')
    print('    docker-compose up -d --force-recreate jupyter')
    print('  Then restart this notebook and re-run from Section 0.')

# 3. Airflow import errors — files that Airflow found but could not parse
print()
print('Airflow import errors:')
resp = requests.get(f'{AIRFLOW_BASE}/api/v1/importErrors', auth=AIRFLOW_AUTH, timeout=10)
if resp.status_code != 200:
    print(f'  Could not fetch (HTTP {resp.status_code})')
else:
    errs = resp.json().get('import_errors', [])
    if not errs:
        print('  None — no parse errors detected')
    else:
        for e in errs:
            print(f'  FILE : {e["filename"]}')
            print(f'  ERROR: {e["stack_trace"][:500]}')
            print()


DAGS_DIR = /home/jovyan/work/dags
Files found:
  lab06_branch_xcom.py  (3511 bytes)
  lab06_hello_dag.py  (2065 bytes)
  lab06_taskflow_demo.py  (2419 bytes)

VOLUME MOUNT: BROKEN  classroom_smoke_test.py NOT found in DAGS_DIR
  Files written here go to ./notebooks/dags/ on the host,
  but Airflow watches ./dags/ — a completely different folder.
  FIX: run from D:\Big_Data_Lab\infrastructure on your laptop:
    docker-compose up -d --force-recreate jupyter
  Then restart this notebook and re-run from Section 0.

Airflow import errors:
  None — no parse errors detected


In [14]:
# ── Unpause our lab DAGs so they can be triggered ───────────────────────────────
# New DAGs are paused by default to prevent accidental runs.
# We send a PATCH request (HTTP verb for partial updates) to set is_paused=False.

for dag_id in ["lab06_hello_dag", "lab06_taskflow_demo", "lab06_branch_xcom"]:
    resp = requests.patch(
        f"{AIRFLOW_BASE}/api/v1/dags/{dag_id}",   # PATCH /api/v1/dags/{dag_id} = update this DAG's settings
        auth=AIRFLOW_AUTH,
        json={"is_paused": False},                 # send JSON body; is_paused=False means 'activate the DAG'
        timeout=10,
    )
    if resp.status_code == 200:     # 200 = success
        print(f"Unpaused: {dag_id}")
    else:                           # anything else = something went wrong
        print(f"Error for {dag_id}: {resp.status_code}  {resp.text}")

Error for lab06_hello_dag: 404  {
  "detail": null,
  "status": 404,
  "title": "Dag with id: 'lab06_hello_dag' not found",
  "type": "https://airflow.apache.org/docs/apache-airflow/2.9.1/stable-rest-api-ref.html#section/Errors/NotFound"
}

Error for lab06_taskflow_demo: 404  {
  "detail": null,
  "status": 404,
  "title": "Dag with id: 'lab06_taskflow_demo' not found",
  "type": "https://airflow.apache.org/docs/apache-airflow/2.9.1/stable-rest-api-ref.html#section/Errors/NotFound"
}

Error for lab06_branch_xcom: 404  {
  "detail": null,
  "status": 404,
  "title": "Dag with id: 'lab06_branch_xcom' not found",
  "type": "https://airflow.apache.org/docs/apache-airflow/2.9.1/stable-rest-api-ref.html#section/Errors/NotFound"
}



In [15]:
# ── Trigger manual DAG runs ──────────────────────────────────────────────────────
# We POST to /dagRuns to start a new run. Airflow assigns it a unique run_id.
# We capture the run_id so we can monitor it in the next cell.

def trigger_dag(dag_id, conf=None):
    """Start a manual DAG run and return its run_id."""
    payload = {
        "conf": conf or {},                       # conf = arbitrary JSON config passed to the DAG
        "logical_date": f"{DATE}T00:00:00Z",      # logical_date = the 'business date' for this run
    }
    resp = requests.post(
        f"{AIRFLOW_BASE}/api/v1/dags/{dag_id}/dagRuns",  # POST = create a new resource (a new run)
        auth=AIRFLOW_AUTH,
        json=payload,    # json= automatically sets Content-Type: application/json and serializes the dict
        timeout=10,
    )
    if resp.status_code in (200, 409):   # 200 = created; 409 = run for this date already exists
        run_id = resp.json().get("dag_run_id", "already_exists")
        print(f"Triggered {dag_id:<35}  run_id = {run_id}")
        return run_id
    else:
        print(f"Error {resp.status_code}: {resp.text}")
        return None

run_id_hello    = trigger_dag("lab06_hello_dag")       # trigger the first DAG
run_id_taskflow = trigger_dag("lab06_taskflow_demo")   # trigger the second DAG
run_id_branch   = trigger_dag("lab06_branch_xcom")     # trigger the third DAG

Error 404: {
  "detail": "DAG with dag_id: 'lab06_hello_dag' not found",
  "status": 404,
  "title": "DAG not found",
  "type": "https://airflow.apache.org/docs/apache-airflow/2.9.1/stable-rest-api-ref.html#section/Errors/NotFound"
}

Error 404: {
  "detail": "DAG with dag_id: 'lab06_taskflow_demo' not found",
  "status": 404,
  "title": "DAG not found",
  "type": "https://airflow.apache.org/docs/apache-airflow/2.9.1/stable-rest-api-ref.html#section/Errors/NotFound"
}

Error 404: {
  "detail": "DAG with dag_id: 'lab06_branch_xcom' not found",
  "status": 404,
  "title": "DAG not found",
  "type": "https://airflow.apache.org/docs/apache-airflow/2.9.1/stable-rest-api-ref.html#section/Errors/NotFound"
}



In [16]:
# ── Poll a DAG run until it finishes ────────────────────────────────────────────
# Airflow runs tasks asynchronously. We must poll the API to check progress.
# This function polls every 'interval' seconds, up to 'max_wait' seconds total.

import time   # time.sleep() = pause execution for N seconds

def poll_dag_run(dag_id, run_id, max_wait=120, interval=5):
    """Poll DAG run state until it reaches 'success' or 'failed', or times out."""
    for attempt in range(max_wait // interval):   # max_wait / interval = number of polling attempts
        resp  = requests.get(
            f"{AIRFLOW_BASE}/api/v1/dags/{dag_id}/dagRuns/{run_id}",  # GET = read the current run status
            auth=AIRFLOW_AUTH,
            timeout=10,
        )
        state = resp.json().get("state", "unknown")   # extract 'state' from the response
        print(f"  [{attempt+1:2d}] {dag_id} => {state}")   # print progress: queued -> running -> success/failed

        if state in ("success", "failed"):   # terminal states — stop polling
            return state

        time.sleep(interval)   # wait before checking again

    return "timeout"   # ran out of polling attempts

if run_id_hello:
    print("=== Polling lab06_hello_dag ===")
    final = poll_dag_run("lab06_hello_dag", run_id_hello)
    print(f"Final state: {final}")

In [18]:
# ── Inspect individual task states for a completed run ──────────────────────────
# A DAG run contains multiple task instances. Each has its own state and duration.

def get_task_instances(dag_id, run_id):
    """Print the state and duration of every task in a DAG run."""
    resp  = requests.get(
        f"{AIRFLOW_BASE}/api/v1/dags/{dag_id}/dagRuns/{run_id}/taskInstances",
        auth=AIRFLOW_AUTH,
        timeout=10,
    )
    tasks = resp.json().get("task_instances", [])

    for t in tasks:
        task_id  = str(t.get('task_id')  or '?')   # str() converts None -> 'None'; 'or' replaces None with '?'
        state    = str(t.get('state')    or '?')   # state can be None if task is still queued
        duration = str(t.get('duration') or '?')   # duration is None until the task finishes
        print(
            f"  task = {task_id:30s}"
            f"  state = {state:12s}"
            f"  duration = {duration}s"
        )

if run_id_hello:
    print("=== Task instances for lab06_hello_dag ===")
    get_task_instances("lab06_hello_dag", run_id_hello)

if run_id_taskflow:
    # Use \n inside the quotes to create a blank line in the console output
    print("\n=== Task instances for lab06_taskflow_demo ===")
    get_task_instances("lab06_taskflow_demo", run_id_taskflow)

if run_id_branch:
    # Same here: \n is the character for "new line"
    print("\n=== Task instances for lab06_branch_xcom ===")
    get_task_instances("lab06_branch_xcom", run_id_branch)


---
## Section 5 — Write the SparkSubmit DAG and PySpark Job

Now we build a **production-style ETL pipeline** DAG that:

```
S3KeySensor         SparkSubmitOperator    PythonOperator     BranchPythonOperator
(wait for file)  ->  (run Spark job)    ->  (validate output) -> (notify success/failure)
```

**S3KeySensor** = waits (polls) until a specific file appears in MinIO  
**SparkSubmitOperator** = submits a PySpark script to the Spark cluster  
**BranchPythonOperator** = routes to success or failure notification based on validation result

In [19]:
# ── Write the PySpark transformation script ──────────────────────────────────────
# This is a standalone Python script that Airflow's SparkSubmitOperator will run.
# It reads raw CSV from MinIO, aggregates by country, and writes Parquet back to MinIO.

pyspark_code = """
#!/usr/bin/env python3
# transform_transactions.py
# Standalone PySpark job — triggered by Airflow SparkSubmitOperator.
# Run manually: spark-submit transform_transactions.py --date 2026-04-27

import argparse   # argparse = parse command-line arguments passed by SparkSubmitOperator
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count, avg, round as _round

def main(date: str):
    spark = (
        SparkSession.builder
        .appName(f'TransformTransactions-{date}')   # name shown in Spark UI for this job
        .config('spark.hadoop.fs.s3a.endpoint',          'http://minio:9000')
        .config('spark.hadoop.fs.s3a.access.key',        'admin')
        .config('spark.hadoop.fs.s3a.secret.key',        'bigdata123')
        .config('spark.hadoop.fs.s3a.path.style.access', 'true')
        .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
        .getOrCreate()
    )

    input_path = f's3a://warehouse/raw/transactions/{date}/data.csv'
    print(f'[INFO] Reading: {input_path}')

    df = spark.read.csv(input_path, header=True, inferSchema=True)
    print(f'[INFO] Total rows: {df.count()}')

    # Step 1: keep only successful transactions
    success_df = df.filter(col('status') == 'SUCCESS')

    # Step 2: group by country and compute summary statistics
    agg_df = (
        success_df
        .groupBy('country')
        .agg(
            count('txn_id').alias('txn_count'),           # how many transactions per country
            _round(_sum('amount'), 2).alias('total_amount'),   # total value processed
            _round(avg('amount'),  2).alias('avg_amount'),     # average transaction size
        )
        .orderBy('country')
    )

    # Write Parquet back to MinIO — partitioned by date for efficient future queries
    output_path = f's3a://warehouse/processed/transactions/date={date}/'
    print(f'[INFO] Writing: {output_path}')
    agg_df.write.mode('overwrite').parquet(output_path)   # overwrite = safe to re-run

    agg_df.show()
    print(f'[INFO] Done — {agg_df.count()} country rows written for date={date}')
    spark.stop()   # release resources when done

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--date', required=True, help='Processing date YYYY-MM-DD')
    args = parser.parse_args()
    main(args.date)
"""

spark_script_path = os.path.join(SPARK_JOBS_DIR, "transform_transactions.py")
with open(spark_script_path, "w") as f:
    f.write(pyspark_code)
print(f"Written : {spark_script_path}")

Written : /home/jovyan/work/spark_jobs/transform_transactions.py


In [20]:
# ── Write the full ETL pipeline DAG ─────────────────────────────────────────────

spark_etl_dag_code = """
# lab06_spark_etl.py  --  AUTO-GENERATED by lab06_airflow.ipynb
#
# Full ETL pipeline:
#   wait_for_csv -> spark_transform -> validate_output -> branch -> notify_success / notify_failure

from datetime import datetime, timedelta
from airflow import DAG
from airflow.providers.apache.spark.operators.spark_submit import SparkSubmitOperator
from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor   # S3KeySensor = wait for a file to appear in S3/MinIO
from airflow.operators.python import PythonOperator, BranchPythonOperator
from airflow.operators.dummy import DummyOperator

# Spark config dict passed to the SparkSubmitOperator
# These settings tell the Spark cluster how to connect to MinIO
SPARK_CONF = {
    'spark.hadoop.fs.s3a.endpoint':          'http://minio:9000',
    'spark.hadoop.fs.s3a.access.key':        'admin',
    'spark.hadoop.fs.s3a.secret.key':        'bigdata123',
    'spark.hadoop.fs.s3a.path.style.access': 'true',
    'spark.hadoop.fs.s3a.impl':              'org.apache.hadoop.fs.s3a.S3AFileSystem',
}

with DAG(
    dag_id='lab06_spark_etl',
    schedule_interval='0 2 * * *',    # cron expression: run at 02:00 every day
    start_date=datetime(2026, 4, 1),
    catchup=False,
    max_active_runs=1,                # prevent overlapping runs if a run is slow
    default_args={
        'owner': 'innocent',
        'retries': 2,
        'retry_delay': timedelta(minutes=5),
        'email_on_failure': False,
        'sla': timedelta(hours=3),    # SLA = Service Level Agreement; alert if task exceeds 3 hours
    },
    tags=['lab06', 'spark', 'etl', 'finance'],
) as dag:

    # Task 1: wait for the input file to appear in MinIO before starting the Spark job
    # {{ ds }} is a Jinja template that Airflow fills in with the execution date (YYYY-MM-DD)
    wait_for_file = S3KeySensor(
        task_id='wait_for_csv',
        bucket_name='warehouse',                             # the MinIO bucket to check
        bucket_key='raw/transactions/{{ ds }}/data.csv',    # the specific key (path) to wait for
        aws_conn_id='minio_s3',         # Airflow Connection ID — configure in Admin -> Connections
        poke_interval=30,               # check every 30 seconds
        timeout=60 * 30,                # give up after 30 minutes
        mode='poke',                    # 'poke' = keep the worker slot occupied while waiting
    )

    # Task 2: submit the PySpark script to the Spark cluster
    spark_transform = SparkSubmitOperator(
        task_id='spark_transform',
        application='/opt/airflow/spark_jobs/transform_transactions.py',  # script path in the Airflow container
        conn_id='spark_default',         # Airflow Connection for Spark — configure in Admin -> Connections
        application_args=['--date', '{{ ds }}'],  # command-line args passed to the script
        conf=SPARK_CONF,                 # Spark config overrides
        num_executors=1,                 # 1 executor is enough for our small dataset
        executor_memory='512m',          # memory per executor
        driver_memory='256m',            # memory for the driver process
    )

    def validate_output(ds=None, **context):
        'Check that the Spark job actually wrote output files to MinIO.'
        import boto3
        s3 = boto3.client('s3', endpoint_url='http://minio:9000',
                          aws_access_key_id='admin', aws_secret_access_key='bigdata123')
        prefix = f'processed/transactions/date={ds}/'   # the output path the Spark job writes to
        resp   = s3.list_objects_v2(Bucket='warehouse', Prefix=prefix)
        files  = resp.get('Contents', [])               # list of file metadata dicts

        if not files:   # no files found — Spark job must have failed
            raise ValueError(f'No output files at s3a://warehouse/{prefix}')

        total_bytes = sum(f['Size'] for f in files)     # total bytes written
        print(f'[VALIDATE] {len(files)} files, {total_bytes} bytes at {prefix}')

        ti = context['ti']                              # task_instance object
        ti.xcom_push(key='file_count', value=len(files))  # push file count for the branch task to read

    validate = PythonOperator(
        task_id='validate_output',
        python_callable=validate_output,
        provide_context=True,   # provide_context=True passes the Airflow context dict to the function
    )

    def branch_on_validation(**context):
        'Read the file count XCom and decide which notification path to take.'
        ti = context['ti']
        fc = ti.xcom_pull(task_ids='validate_output', key='file_count')   # pull the value we pushed above
        return 'notify_success' if (fc and fc > 0) else 'notify_failure'  # return the task_id to run next

    branch         = BranchPythonOperator(task_id='branch',          python_callable=branch_on_validation)
    notify_success = DummyOperator(task_id='notify_success')   # in production: send Slack/email success alert
    notify_failure = DummyOperator(task_id='notify_failure')   # in production: page on-call engineer

    # Wire up the pipeline
    wait_for_file >> spark_transform >> validate >> branch
    branch >> [notify_success, notify_failure]
"""

dag_path = os.path.join(DAGS_DIR, "lab06_spark_etl.py")
with open(dag_path, "w") as f:
    f.write(spark_etl_dag_code)
print(f"Written : {dag_path}")

Written : /home/jovyan/work/dags/lab06_spark_etl.py


---
## Section 6 — Run the Spark Job Directly (In-Notebook Test)

Before triggering via Airflow, we validate the transformation logic here in the notebook.  
This is good practice: test the core logic first, then wire it into the orchestrator.

In [21]:
# ── Run the ETL transformation directly in the notebook ─────────────────────────
from pyspark.sql.functions import col, count, avg
from pyspark.sql.functions import sum   as _sum    # rename to avoid shadowing Python's built-in sum()
from pyspark.sql.functions import round as _round  # rename to avoid shadowing Python's built-in round()

# Step 1: Read the raw CSV we uploaded in Section 2
raw_df = spark.read.csv(
    f"s3a://{MINIO_BUCKET}/raw/transactions/{DATE}/data.csv",
    header=True,       # first row = column names
    inferSchema=True,  # auto-detect column types
)
print(f"Raw row count : {raw_df.count()}")   # should be 1,000
raw_df.printSchema()                          # shows column names and detected types

# Step 2: Filter — keep only transactions that succeeded
success_df = raw_df.filter(col("status") == "SUCCESS")   # col("status") = reference the status column
print(f"Success rows  : {success_df.count()}")

# Step 3: Aggregate — one summary row per country
agg_df = (
    success_df
    .groupBy("country")
    .agg(
        count("txn_id").alias("txn_count"),             # count transactions per country
        _round(_sum("amount"), 2).alias("total_amount"),# sum of all transaction amounts
        _round(avg("amount"),  2).alias("avg_amount"),  # average transaction size
    )
    .orderBy("country")   # sort alphabetically for readability
)

print("\nAggregated result (5 rows = 5 countries):")
agg_df.show()

Raw row count : 1000
root
 |-- txn_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- sender: long (nullable = true)
 |-- receiver: long (nullable = true)
 |-- country: string (nullable = true)
 |-- status: string (nullable = true)
 |-- txn_date: date (nullable = true)

Success rows  : 748

Aggregated result (5 rows = 5 countries):
+-------+---------+------------+----------+
|country|txn_count|total_amount|avg_amount|
+-------+---------+------------+----------+
|     ET|      170|  7819020.94|  45994.24|
|     KE|      150|  8080443.01|  53869.62|
|     RW|      143|  7499863.66|   52446.6|
|     TZ|      142|  6999078.51|  49289.29|
|     UG|      143|  6546687.51|  45781.03|
+-------+---------+------------+----------+



In [22]:
# ── Write the aggregated result to MinIO as Parquet ──────────────────────────────
# Parquet is a columnar binary format — much faster to query than CSV for analytics.
# date=DATE in the path creates a Hive-style partition that enables efficient date filtering.

output_path = f"s3a://{MINIO_BUCKET}/processed/transactions/date={DATE}/"

agg_df.write \
    .mode("overwrite") \
    .parquet(output_path)   # parquet() = write in Parquet format (NOT CSV)

print(f"Written {agg_df.count()} rows  ->  {output_path}")

Written 5 rows  ->  s3a://warehouse/processed/transactions/date=2026-04-27/


In [23]:
# ── Verify: list the Parquet files that Spark created in MinIO ────────────────────
# Spark writes one file per partition. With small data and local[*] mode this is typically 1-2 files.

prefix = f"processed/transactions/date={DATE}/"   # the folder path inside the bucket
resp   = s3.list_objects_v2(Bucket=MINIO_BUCKET, Prefix=prefix)   # list all files under this prefix

print(f"Files in MinIO at s3a://{MINIO_BUCKET}/{prefix}:")
for obj in resp.get("Contents", []):   # Contents = list of file info dicts
    print(f"  {obj['Key']}  ({obj['Size']:,} bytes)")

Files in MinIO at s3a://warehouse/processed/transactions/date=2026-04-27/:
  processed/transactions/date=2026-04-27/_SUCCESS  (0 bytes)
  processed/transactions/date=2026-04-27/part-00000-751cb5d6-abff-4bcb-a4c1-defc667b7b46-c000.snappy.parquet  (1,404 bytes)


In [24]:
# ── Read back the Parquet files to confirm the data is correct ───────────────────
# This is the 'round-trip test': write -> read back -> verify the content matches.

result_df = spark.read.parquet(output_path)   # read all Parquet part-files in the folder

print(f"Output row count : {result_df.count()}")   # should be 5 (one per country)
result_df.show()

Output row count : 5
+-------+---------+------------+----------+
|country|txn_count|total_amount|avg_amount|
+-------+---------+------------+----------+
|     ET|      170|  7819020.94|  45994.24|
|     KE|      150|  8080443.01|  53869.62|
|     RW|      143|  7499863.66|   52446.6|
|     TZ|      142|  6999078.51|  49289.29|
|     UG|      143|  6546687.51|  45781.03|
+-------+---------+------------+----------+



---
## Section 7 — Trigger the Spark ETL DAG via Airflow

Now that we have confirmed the Spark job works correctly in the notebook,  
we trigger it through Airflow so the full orchestration pipeline runs end-to-end.

In [25]:
# ── Wait for Airflow to detect the new DAG, then trigger it ─────────────────────

print("Waiting 35 seconds for Airflow Scheduler to parse lab06_spark_etl.py...")
time.sleep(35)   # the scheduler polls the dags folder every ~30 seconds by default

# Unpause the spark ETL DAG (new DAGs start paused)
resp = requests.patch(
    f"{AIRFLOW_BASE}/api/v1/dags/lab06_spark_etl",
    auth=AIRFLOW_AUTH,
    json={"is_paused": False},   # activate the DAG
    timeout=10,
)
print(f"Unpause response: {resp.status_code}")

# Trigger a manual run
run_id_spark = trigger_dag("lab06_spark_etl")   # reuse our trigger_dag helper from Section 4
print(f"Run ID: {run_id_spark}")
print("\nNote: S3KeySensor will pass immediately (file already uploaded in Section 2).")
print("SparkSubmitOperator may take 1-3 minutes to complete.")

Waiting 35 seconds for Airflow Scheduler to parse lab06_spark_etl.py...
Unpause response: 404
Error 404: {
  "detail": "DAG with dag_id: 'lab06_spark_etl' not found",
  "status": 404,
  "title": "DAG not found",
  "type": "https://airflow.apache.org/docs/apache-airflow/2.9.1/stable-rest-api-ref.html#section/Errors/NotFound"
}

Run ID: None

Note: S3KeySensor will pass immediately (file already uploaded in Section 2).
SparkSubmitOperator may take 1-3 minutes to complete.


In [29]:
# ── Poll the ETL DAG run until it completes ──────────────────────────────────────
# We use a longer max_wait (300s) and bigger interval (10s) because Spark jobs take longer than simple Python tasks.

if run_id_spark:
    print("=== Monitoring lab06_spark_etl ===")
    final_state = poll_dag_run(
        "lab06_spark_etl",
        run_id_spark,
        max_wait=300,   # wait up to 5 minutes
        interval=10,    # check every 10 seconds
    )
    print(f"\nFinal DAG state: {final_state}")

In [30]:
# ── Print the state of each task in the completed ETL run ────────────────────────
if run_id_spark:
    print("=== Task states for lab06_spark_etl ===")
    get_task_instances("lab06_spark_etl", run_id_spark)   # reuse helper from Section 4

---
## Section 8 — Airflow UI Navigation Guide

Open **http://localhost:8090** (admin / admin) and explore these views:

| View | How to Access | What to Look For |
|------|---------------|------------------|
| **DAG List** | Home page | Toggle pause/active; coloured run-state squares |
| **Grid View** | Click DAG name | All runs as columns; click a task cell for logs |
| **Graph View** | Grid → Graph tab | Visual task topology with colour-coded states |
| **Gantt Chart** | Grid → Gantt tab | Task timing and queue wait times |
| **Task Log** | Grid → click cell → Log | Full stdout/stderr of the task |
| **XCom** | Grid → click cell → XCom | Key-value pairs pushed by `xcom_push` |
| **SLA Misses** | Admin → SLA Misses | Tasks that took longer than their SLA |
| **Connections** | Admin → Connections | Configure `minio_s3` and `spark_default` |
| **Variables** | Admin → Variables | Runtime key-value configuration |

### Configuring Connections for the ETL DAG

The `lab06_spark_etl` DAG uses two Airflow Connections that you must create manually in the UI:

**1. `minio_s3` — for the S3KeySensor:**  
> Admin → Connections → + → Conn Type: Amazon Web Services  
> Conn ID: `minio_s3`  
> Extra: `{"endpoint_url": "http://minio:9000", "aws_access_key_id": "admin", "aws_secret_access_key": "bigdata123"}`

**2. `spark_default` — for the SparkSubmitOperator:**  
> Admin → Connections → + → Conn Type: Spark  
> Conn ID: `spark_default`  
> Host: `spark://spark-master`  
> Port: `7077`

---
## Section 9 — Cleanup

In [ ]:
# ── Stop the SparkSession to release memory and CPU ──────────────────────────────
# Always stop Spark when you are done — it holds RAM even when idle.
spark.stop()   # .stop() gracefully shuts down the SparkContext and frees all resources
print("SparkSession stopped.")

---
## Summary

In this lab you:

1. **Verified** all Docker services are healthy (Airflow, Spark, MinIO, PostgreSQL)
2. **Uploaded** synthetic mobile-money data to MinIO using `boto3`
3. **Wrote three DAG files** from Python — Hello, TaskFlow, and Branch/XCom
4. **Triggered and monitored** DAG runs via the Airflow REST API
5. **Validated** in-notebook PySpark transformations reading from and writing to MinIO
6. **Orchestrated** a full ETL pipeline: S3KeySensor → SparkSubmit → Validate → Branch

---
**Assignment due: Sunday, 3 May 2026, 23:59 EAT**  
Submit to `innocent@iitmz.ac.in` — subject: `[Z5008-Lab06] YourName_RegNo`